# Yellow Traffic-Sign Segmentation

This notebook displays the filename and seven processing stages for every traffic-sign image.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


VALID_EXTENSIONS = (
    ".jpg", ".jpeg", ".png",
    ".bmp", ".tif", ".tiff"
)

# Load valid images from the folder
def load_images(img_dir):
    if not os.path.isdir(img_dir):
        raise FileNotFoundError(f"Image folder not found: {img_dir}")

    images = []

    for filename in sorted(os.listdir(img_dir)):
        if not filename.lower().endswith(VALID_EXTENSIONS):
            continue

        image_path = os.path.join(img_dir, filename)
        image = cv2.imread(image_path)

        if image is None:
            print(f"Skipped unreadable image: {filename}")
            continue

        images.append((filename, image))

    return images


# Enhance dark images using CLAHE
def clean_dark_image(
    image,
    brightness_threshold=140,
    clahe_clip_limit=2.0,
    clahe_grid_size=(8, 8)
):
    cleaned_image = image.copy()

    hsv_image = cv2.cvtColor(cleaned_image, cv2.COLOR_BGR2HSV)
    h_channel, s_channel, v_channel = cv2.split(hsv_image)

    height, width = v_channel.shape

    # Check the brightness near the image centre
    centre_region = v_channel[
        height // 4: 3 * height // 4,
        width // 4: 3 * width // 4
    ]

    mean_brightness = float(np.mean(centre_region))

    # Enhance dark images using CLAHE
    if mean_brightness < brightness_threshold:
        clahe = cv2.createCLAHE(
            clipLimit=clahe_clip_limit,
            tileGridSize=clahe_grid_size
        )

        enhanced_v = clahe.apply(v_channel)
        enhanced_hsv = cv2.merge(
            (h_channel, s_channel, enhanced_v)
        )

        cleaned_image = cv2.cvtColor(
            enhanced_hsv,
            cv2.COLOR_HSV2BGR
        )

    return cleaned_image


# Set a valid block size for adaptive thresholding
def get_valid_block_size(
    image, 
    preferred_size=51
):
    block_size = min(
        preferred_size,
        min(image.shape[:2])
    )

    if block_size % 2 == 0:
        block_size -= 1

    if block_size < 3:
        raise ValueError(
            "The image is too small for adaptive thresholding."
        )

    return block_size


# Create the combined yellow mask
def create_combined_threshold(
    cleaned_image,
    lower_yellow=(5, 80, 30),
    upper_yellow=(40, 255, 255),
    adaptive_block_size=51,
    adaptive_c=7
):
    hsv_image = cv2.cvtColor(
        cleaned_image,
        cv2.COLOR_BGR2HSV
    )

    _, _, v_channel = cv2.split(hsv_image)
    
    lower_bound = np.array(
        lower_yellow,
        dtype=np.uint8
    )

    upper_bound = np.array(
        upper_yellow,
        dtype=np.uint8
    )

    # Extract yellow pixels using the HSV range
    yellow_mask = cv2.inRange(
        hsv_image,
        lower_bound,
        upper_bound
    )

    block_size = get_valid_block_size(
        v_channel,
        adaptive_block_size
    )

    # Apply adaptive thresholding for uneven lighting
    adaptive_image = cv2.adaptiveThreshold(
        v_channel,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        block_size,
        adaptive_c
    )

    # Combine the colour and threshold masks
    combined_threshold = cv2.bitwise_and(
        yellow_mask,
        adaptive_image
    )

    return combined_threshold


# Clean the mask using morphological closing
def apply_morphological_closing(
    combined_threshold,
    kernel_size=(3, 3),
    iterations=1
):
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        kernel_size
    )

    # Connect broken regions and fill small gaps
    morphological_image = cv2.morphologyEx(
        combined_threshold,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=iterations
    )

    return morphological_image


# Select the best traffic-sign contour
def select_best_contour(
    morphological_image,
    min_area_ratio=0.005,
    max_area_ratio=0.75
):
    height, width = morphological_image.shape
    image_area = height * width

    image_centre_x = width / 2
    image_centre_y = height / 2

    # Find the outer contours
    contours, _ = cv2.findContours(
        morphological_image.copy(),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    best_contour = None
    best_score = -1

    for contour in contours:
        contour_area = cv2.contourArea(contour)

        # Remove contours that are too small or too large
        if not (
            image_area * min_area_ratio
            <= contour_area
            <= image_area * max_area_ratio
        ):
            continue

        moments = cv2.moments(contour)

        if moments["m00"] == 0:
            continue

        contour_centre_x = (
            moments["m10"] / moments["m00"]
        )

        contour_centre_y = (
            moments["m01"] / moments["m00"]
        )

        centre_distance = np.hypot(
            contour_centre_x - image_centre_x,
            contour_centre_y - image_centre_y
        )

        # Prefer a large contour near the image centre
        contour_score = (
            contour_area / (centre_distance + 1)
        )

        if contour_score > best_score:
            best_contour = contour
            best_score = contour_score

    return best_contour


# Reconstruct an incomplete outer contour
def reconstruct_outer_contour(
    selected_contour,
    solidity_threshold=0.90,
    approximation_ratio=0.01
):
    if selected_contour is None:
        return None

    contour_area = cv2.contourArea(
        selected_contour
    )

    # Create a convex hull around the contour
    convex_hull = cv2.convexHull(
        selected_contour
    )

    hull_area = cv2.contourArea(
        convex_hull
    )

    if hull_area == 0:
        return selected_contour

    solidity = contour_area / hull_area

    # Reconstruct the boundary if the contour is incomplete
    if solidity < solidity_threshold:
        hull_perimeter = cv2.arcLength(
            convex_hull,
            True
        )

        return cv2.approxPolyDP(
            convex_hull,
            approximation_ratio * hull_perimeter,
            True
        )

    return selected_contour


# Create the contour and segmentation outputs
def create_contour_outputs(
    original_image,
    selected_contour
):
    # Create an image for displaying the green contour
    contour_only_image = np.zeros_like(
        original_image
    )

    # Create the filled contour mask
    contour_mask = np.zeros(
        original_image.shape[:2],
        dtype=np.uint8
    )

    # Return empty results if no contour is found
    if selected_contour is None:

        segmented_original = np.zeros_like(
            original_image
        )

        return (
            contour_only_image,
            contour_mask,
            segmented_original
        )

    # Reconstruct the outer boundary
    outer_contour = reconstruct_outer_contour(
        selected_contour
    )

    # Draw the detected contour
    cv2.drawContours(
        contour_only_image,
        [outer_contour],
        -1,
        (0, 255, 0),
        2
    )

    # Fill the detected contour
    cv2.drawContours(
        contour_mask,
        [outer_contour],
        -1,
        255,
        cv2.FILLED
    )

    # Extract the traffic-sign region
    segmented_original = cv2.bitwise_and(
        original_image,
        original_image,
        mask=contour_mask
    )

    return (
        contour_only_image,
        contour_mask,
        segmented_original
    )


# Process all images in the folder
def process_folder(
    img_dir,
    lower_yellow=(5, 80, 30),
    upper_yellow=(40, 255, 255),
    brightness_threshold=140,
    adaptive_block_size=51,
    adaptive_c=7,
    close_kernel_size=(3, 3),
    close_iterations=1,
    min_area_ratio=0.005,
    max_area_ratio=0.75
):
    images = load_images(img_dir)
    results = []

    # Process each image
    for filename, original_image in images:
        cleaned_image = clean_dark_image(
            original_image,
            brightness_threshold=brightness_threshold
        )

        combined_threshold = create_combined_threshold(
            cleaned_image,
            lower_yellow=lower_yellow,
            upper_yellow=upper_yellow,
            adaptive_block_size=adaptive_block_size,
            adaptive_c=adaptive_c
        )

        morphological_image = apply_morphological_closing(
            combined_threshold,
            kernel_size=close_kernel_size,
            iterations=close_iterations
        )

        selected_contour = select_best_contour(
            morphological_image,
            min_area_ratio=min_area_ratio,
            max_area_ratio=max_area_ratio
        )

        (
            contour_only_image,
            contour_mask,
            segmented_original
        ) = create_contour_outputs(
            original_image,
            selected_contour
        )

        # Store the processing results
        results.append({
            "filename": filename,
            "original": original_image,
            "morphological": morphological_image,
            "detected_contour": contour_only_image,
            "contour_mask": contour_mask,
            "segmented_original": segmented_original
        })

    print(f"Processed {len(results)} images.")

    return results

    
# Display final results for each traffic-sign image
def display_pipeline_results(results):

    if len(results) == 0:
        print("No results available.")
        return

    stages = [
        ("original", "Original Image"),
        ("morphological", "Final Mask"),
        ("detected_contour", "Detected Contour"),
        ("contour_mask", "Contour Mask"),
        ("segmented_original", "Segmented Image")
    ]

    grayscale_stages = [
        "morphological",
        "contour_mask"
    ]

    for result in results:

        filename = result["filename"]

        # Create one separate figure for every input image
        fig, axes = plt.subplots(
            1,
            5,
            figsize=(18, 4)
        )

        for axis, (result_key, stage_name) in zip(
            axes,
            stages
        ):

            image = result[result_key]

            if result_key in grayscale_stages:
                axis.imshow(
                    image,
                    cmap="gray",
                    vmin=0,
                    vmax=255
                )

            else:
                image_rgb = cv2.cvtColor(
                    image,
                    cv2.COLOR_BGR2RGB
                )

                axis.imshow(image_rgb)

            axis.set_title(
                stage_name,
                fontsize=11,
                fontweight="bold"
            )

            axis.axis("off")

        fig.suptitle(
            filename,
            fontsize=15,
            fontweight="bold"
        )

        plt.tight_layout(rect=[0, 0, 1, 0.94])

        plt.show()


## Run the segmentation

Place the `yellow_signs` folder in the same location as this notebook, or change `img_dir` below.

In [ ]:
# Run the segmentation process
img_dir = "yellow_signs/"

results = process_folder(img_dir)

display_pipeline_results(results)
